# DWBI Week 4 — Dimensional Modeling
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's Week-3 star (1 fact + 5 dims, 34 rows, 578,650 kg) faces the snowflake question: should DimCity be normalized? Seven tasks count the joins, build the snowflake, do the storage arithmetic, register the relationships, settle cross-filter direction — and the totals do not move by a single kilogram.

## T2 — Snowflake it, then justify it (15 marks)

> Built **without opening `data/snowflake/` first** — the three tables below were derived from the flat
> DimCity alone, then verified against the given snowflake files.

### Part A — DimProvince (6 rows)

| ProvinceKey | ProvinceName | RegionName | CountryKey |
|---|---|---|---|
| 1 | Sindh | South | 1 |
| 2 | Balochistan | South | 1 |
| 3 | Punjab | North | 1 |
| 4 | Islamabad ICT | North | 1 |
| 5 | Khyber Pakhtunkhwa | North | 1 |
| -1 | Unknown | Unknown | -1 |

One row per distinct province (5 + Unknown), surrogate keys minted 1–5 in first-seen order.

### Part B — DimCountry (2 rows)

| CountryKey | CountryName | RegionGroup |
|---|---|---|
| 1 | Pakistan | South Asia |
| -1 | Unknown | Unknown |

Look hard at how many rows this table has: **one** real country. A whole table, a whole join, for one row.

### Part C — DimCity normalized (15 rows)

Same 15 rows. `Province` and `Region` removed, `ProvinceKey` added:

| CityKey | CityName | ProvinceKey | IsPortCity | FacilityType |
|---|---|---|---|---|
| 1 | Karachi | 1 | Yes | Sea Port + Rail Hub |
| 2 | Gwadar | 2 | Yes | Sea Port |
| 3 | Lahore | 3 | No | Dry Port |
| … | … | … | … | … |
| -1 | Unknown | -1 | No | Unknown |

(Full 15-row table in the workbook, sheet T2.)

![Flat → snowflake](charts/t2_snowflake.png)

### Part D — the verdict (the reason carries the marks)

> **FLATTEN.** T1 shows only 4 of 8 questions change (+5 joins on a 15-row dimension that fits in
> cache), and T3 shows the storage saving is noise (−18 bytes at 15 rows; 0.1% of the fact table
> even at 50M rows). The joins cost more than the bytes save.

In [ ]:
# T2: derive the snowflake from flat DimCity WITHOUT opening data/snowflake/
import pandas as pd
city = pd.read_csv("data/star/DimCity.csv")
provs = city[["Province","Region"]].drop_duplicates().reset_index(drop=True)
print("distinct provinces:", len(provs[provs.Province!="Unknown"]), "+ Unknown")
# mint keys in first-seen order
keymap = {p:i+1 for i,p in enumerate(provs[provs.Province!="Unknown"]["Province"])}
keymap["Unknown"] = -1
norm = city[["CityKey","CityName","IsPortCity","FacilityType"]].copy()
norm["ProvinceKey"] = city["Province"].map(keymap)
print("normalized DimCity:", norm.shape, "| columns:", list(norm.columns))
assert list(norm.columns)==["CityKey","CityName","IsPortCity","FacilityType","ProvinceKey"]
assert (norm["ProvinceKey"]!=-1).sum()==14 and (norm["ProvinceKey"]==-1).sum()==1
print("Part A/B/C derived OK — Province/Region removed, ProvinceKey added")

In [ ]:
# T2 verification: compare against the given snowflake files
import pandas as pd
mine_prov = pd.DataFrame({"ProvinceKey":[1,2,3,4,5,-1],
    "ProvinceName":["Sindh","Balochistan","Punjab","Islamabad ICT","Khyber Pakhtunkhwa","Unknown"],
    "RegionName":["South","South","North","North","North","Unknown"],"CountryKey":[1,1,1,1,1,-1]})
given_prov = pd.read_csv("data/snowflake/DimProvince.csv")
print("DimProvince matches:", mine_prov.equals(given_prov))
given_city = pd.read_csv("data/snowflake/DimCity_Snowflaked.csv")
print("snowflaked DimCity rows:", len(given_city), "| ProvinceKey present, Province/Region gone:",
      "ProvinceKey" in given_city.columns and "Province" not in given_city.columns)
assert mine_prov.equals(given_prov)